# Amazon ML Challenge: cross-encoder round 2 (Kaggle, GPU T4 ×2)

One notebook, two runs. Set `RUN` in the first code cell.

| RUN | What it does | Inputs to attach | Time |
|---|---|---|---|
| **"A"** | score the full pair set with the **existing small model** (no training) | dataset `ce_handoff2` **and** the output of the previous committed notebook (the one with `ce_model/`) | ~1–1.5 h |
| **"B"** | train **multilingual-e5-base** on the bigger set, then score the full pair set | dataset `ce_handoff2` only | ~7 h (training capped by `CE_TRAIN_MINUTES`) |

**Setup:** Settings → Accelerator **GPU T4 x2**, **Internet on**. Then **Save Version → Save & Run All (Commit)**.
Outputs appear in the Output tab when the run ends: `ce_val_full_scores_<tag>.parquet` and
`ce_test_full_scores_<tag>.parquet` (the tag is `small` or `base`). Run B also saves `ce_model/`.
If a GPU fails (e.g. "uncorrectable ECC error"), stop the session and commit again to get a new machine.

Models: `intfloat/multilingual-e5-small` (MIT, 118M) and `intfloat/multilingual-e5-base` (MIT, 278M).

In [ ]:
RUN = "A"        # "A" = score with the existing small model, "B" = train e5-base and score

import glob
import os
import subprocess
import time

CFG = {"CE_MAXLEN": "96", "CE_INF_BS": "512", "CE_WORKERS": "1", "CE_EPOCHS": "1", "CE_SAVE_EVERY": "1500"}
if RUN == "B":
    CFG |= {"CE_MODEL": "intfloat/multilingual-e5-base", "CE_TAG": "base", "CE_BS": "64", "CE_LR": "3e-5",
            "CE_TRAIN_FILE": "ce_train_big.parquet",
            "CE_MAX_TRAIN": "3500000",     # pairs used for training (the file has 5.7M); lower it if the ETA is too long
            "CE_TRAIN_MINUTES": "240",     # hard cap on training time, so scoring still fits in the 12 h session
            "CE_INF_BS": "384"}
else:
    CFG |= {"CE_MODEL": "intfloat/multilingual-e5-small", "CE_TAG": "small"}

hits = glob.glob("/kaggle/input/**/ce_val_full.parquet", recursive=True)
assert hits, "attach the dataset with ce_val_full / ce_test_full / ce_train_big .parquet (local kaggle/ce_handoff2)"
CFG["CE_DATA"] = os.path.dirname(hits[0])
CFG["CE_OUT"] = "/kaggle/working"
if RUN == "A":
    prev = glob.glob("/kaggle/input/**/ce_model/config.json", recursive=True)
    assert prev, "RUN A needs the previous notebook's output (with ce_model/) attached as an input"
    CFG["CE_MODEL_DIR"] = os.path.dirname(prev[0])
else:
    CFG["CE_MODEL_DIR"] = "/kaggle/working/ce_model"
os.environ.update(CFG)
os.environ.update({"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1", "TOKENIZERS_PARALLELISM": "false"})
print(CFG)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
NGPU = torch.cuda.device_count()
print("GPUs visible:", NGPU)

In [ ]:
%%writefile /kaggle/working/ce.py
"""Cross-encoder: fine-tune (DDP) and score pairs (one shard per GPU).
Tokenisation runs in DataLoader workers (overlaps with the GPU); training batches are length-bucketed.
usage: torchrun --nproc_per_node=N ce.py train | infer <name>"""
import os
import sys
import time

import numpy as np
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

E = os.environ
MODEL, DATA, OUT = E["CE_MODEL"], E["CE_DATA"], E["CE_OUT"]
MODEL_DIR = E.get("CE_MODEL_DIR", os.path.join(OUT, "ce_model"))
TRAIN_FILE = E.get("CE_TRAIN_FILE", "ce_train.parquet")
MAXLEN, BS, INF_BS = int(E.get("CE_MAXLEN", 96)), int(E.get("CE_BS", 128)), int(E.get("CE_INF_BS", 512))
EPOCHS, LR, MAX_TRAIN = float(E.get("CE_EPOCHS", 1)), float(E.get("CE_LR", 6e-5)), int(E.get("CE_MAX_TRAIN", 0))
WORKERS, SAVE_EVERY = int(E.get("CE_WORKERS", 1)), int(E.get("CE_SAVE_EVERY", 1500))
TRAIN_MIN = float(E.get("CE_TRAIN_MINUTES", 0))       # 0 = no time cap


def setup():
    world = int(E.get("WORLD_SIZE", 1))
    if world > 1:
        dist.init_process_group("nccl" if torch.cuda.is_available() and os.name != "nt" else "gloo")
    rank, local = int(E.get("RANK", 0)), int(E.get("LOCAL_RANK", 0))
    dev = torch.device(f"cuda:{local}" if torch.cuda.is_available() else "cpu")
    if dev.type == "cuda":
        torch.cuda.set_device(dev)
    return rank, world, dev


def log(rank, *a):
    if rank == 0:
        print(time.strftime("%H:%M:%S"), *a, flush=True)


class Batches(Dataset):
    """Item i = one pre-planned batch of row indices -> tokenised tensors (runs in a worker process)."""
    def __init__(self, tbl, batches, tok, y=None):
        self.tbl, self.batches, self.tok, self.y = tbl, batches, tok, y

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        idx = np.sort(self.batches[i])
        sub = self.tbl.take(idx)
        x = dict(self.tok(sub.column("r_text").to_pylist(), sub.column("s_text").to_pylist(),
                          truncation="longest_first", max_length=MAXLEN, padding=True, return_tensors="pt"))
        x["_idx"] = torch.from_numpy(idx)
        if self.y is not None:
            x["_y"] = torch.from_numpy(self.y[idx])
        return x


def loader(ds):
    return DataLoader(ds, batch_size=None, shuffle=False, num_workers=WORKERS, pin_memory=True,
                      prefetch_factor=8 if WORKERS else None)


def lengths(tbl):
    return (pc.utf8_length(tbl.column("r_text")).to_numpy(zero_copy_only=False)
            + pc.utf8_length(tbl.column("s_text")).to_numpy(zero_copy_only=False))


def save(model, tok, world, note):
    m = model.module if world > 1 else model
    m.save_pretrained(MODEL_DIR)
    tok.save_pretrained(MODEL_DIR)
    open(os.path.join(MODEL_DIR, "STATUS"), "w").write(note)


def train(rank, world, dev):
    tbl = pq.read_table(os.path.join(DATA, TRAIN_FILE), columns=["r_text", "s_text", "label"])
    if MAX_TRAIN:
        tbl = tbl.slice(0, MAX_TRAIN)
    n = tbl.num_rows
    y = tbl.column("label").to_numpy().astype(np.float32)
    ln = lengths(tbl)
    log(rank, f"model {MODEL}  train pairs {n:,}  positives {int(y.sum()):,}  world={world}")
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).to(dev)
    if world > 1:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[dev.index] if dev.type == "cuda" else None)
    per_ep = n // (BS * world)
    total = int(per_ep * EPOCHS)
    # batch plan: shuffle, bucket by length inside blocks of 64 batches, shuffle the batch order;
    # identical on every rank, rank r takes every world-th batch
    plan = []
    for ep in range(int(np.ceil(EPOCHS))):
        rng = np.random.default_rng(ep)
        perm = rng.permutation(n)[: per_ep * BS * world]
        blk, bs = BS * 64, []
        for i in range(0, len(perm), blk):
            b = perm[i:i + blk]
            b = b[np.argsort(ln[b], kind="stable")]
            bs += [b[j:j + BS] for j in range(0, len(b), BS)]
        plan += [bs[k] for k in rng.permutation(len(bs))]
    plan = plan[: total * world][rank::world]
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    lossf = torch.nn.BCEWithLogitsLoss()
    model.train()
    t0, run = time.time(), 0.0
    for step, x in enumerate(loader(Batches(tbl, plan, tok, y)), 1):
        x.pop("_idx")
        yt = x.pop("_y").to(dev, non_blocking=True)
        x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}
        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            logit = model(**x).logits.squeeze(-1)
        loss = lossf(logit.float(), yt)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        if step % 50 == 0 or step == 1:        # .item() syncs the GPU, so only occasionally
            run = loss.item() if step == 1 else 0.8 * run + 0.2 * loss.item()
            if TRAIN_MIN:                      # time cap: every rank stops at the same step
                over = torch.tensor([float(time.time() - t0 > TRAIN_MIN * 60)], device=dev)
                if world > 1:
                    dist.all_reduce(over, op=dist.ReduceOp.MAX)
                if over.item() > 0:
                    log(rank, f"time cap reached at step {step}/{total}")
                    break
        if step % 200 == 0 or step == total:
            el = time.time() - t0
            log(rank, f"step {step}/{total} loss {run:.4f}  {step * BS * world / el:,.0f} pairs/s  "
                      f"eta {(total - step) * el / step / 60:.1f} min")
        if rank == 0 and step % SAVE_EVERY == 0 and step < total:
            save(model, tok, world, f"partial {step}/{total}")
    if rank == 0:
        save(model, tok, world, "complete")
        log(rank, f"saved {MODEL_DIR}  ({(time.time() - t0) / 60:.1f} min)")


@torch.no_grad()
def infer(rank, world, dev, name):
    n = pq.ParquetFile(os.path.join(DATA, f"{name}.parquet")).metadata.num_rows
    lo, hi = n * rank // world, n * (rank + 1) // world
    tbl = pq.read_table(os.path.join(DATA, f"{name}.parquet"), columns=["r_text", "s_text"]).slice(lo, hi - lo)
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(dev).eval()
    if dev.type == "cuda":
        model = model.half()
    order = np.argsort(lengths(tbl), kind="stable")          # length-sorted batches: little padding
    plan = [order[i:i + INF_BS] for i in range(0, len(order), INF_BS)]
    out = np.zeros(len(order), dtype=np.float32)
    t0 = time.time()
    for b, x in enumerate(loader(Batches(tbl, plan, tok))):
        idx = x.pop("_idx").numpy()
        x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}
        out[idx] = model(**x).logits.squeeze(-1).float().cpu().numpy()
        if b and b % 1000 == 0:
            el = time.time() - t0
            log(rank, f"{name}: {b * INF_BS:,}/{len(order):,} per GPU  {world * b * INF_BS / el:,.0f} pairs/s total  "
                      f"eta {(len(plan) - b) * el / b / 60:.1f} min")
    np.save(os.path.join(OUT, f"scores_{name}_{rank:02d}.npy"), out)
    log(rank, f"{name}: done in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    rank, world, dev = setup()
    if sys.argv[1] == "train":
        train(rank, world, dev)
    else:
        infer(rank, world, dev, sys.argv[2])
    if world > 1:
        dist.barrier()
        dist.destroy_process_group()

In [ ]:
def run(*args):
    cmd = ["torchrun", f"--nproc_per_node={max(NGPU, 1)}", "/kaggle/working/ce.py", *args]
    print(" ".join(cmd), flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Warning" not in line and "warn(" not in line:
            print(line, end="", flush=True)
    assert p.wait() == 0, f"{args} failed"
    print(f"--- {args} took {(time.time() - t) / 60:.1f} min")


status = os.path.join(os.environ["CE_MODEL_DIR"], "STATUS")
if RUN == "B" and not os.path.exists(status):
    run("train")
else:
    print("using model", os.environ["CE_MODEL_DIR"], "->", open(status).read() if os.path.exists(status) else "complete")

In [ ]:
import numpy as np
import pandas as pd

TAG = os.environ["CE_TAG"]


def merge(name, cols):
    df = pd.read_parquet(os.path.join(CFG["CE_DATA"], f"{name}.parquet"), columns=cols)
    parts = sorted(glob.glob(f"/kaggle/working/scores_{name}_*.npy"))
    df["ce_logit"] = np.concatenate([np.load(p) for p in parts])
    df.to_parquet(f"/kaggle/working/{name}_scores_{TAG}.parquet", index=False)
    for p in parts:
        os.remove(p)
    return df


# validation first (small, and it is what we check), then test
for name, cols in (("ce_val_full", ["rid", "s1", "country", "label", "xgb_p"]), ("ce_test_full", ["rid", "s1", "country"])):
    if not os.path.exists(f"/kaggle/working/{name}_scores_{TAG}.parquet"):
        run("infer", name)
        merge(name, cols)
print([f for f in os.listdir("/kaggle/working") if f.endswith(".parquet")])

## Quick check on validation (the real macro-F0.5 evaluation runs locally in `collective.py`)

In [ ]:
from sklearn.metrics import roc_auc_score

val = pd.read_parquet(f"/kaggle/working/ce_val_full_scores_{TAG}.parquet")
print(f"[{TAG}] AUC  xgb {roc_auc_score(val.label, val.xgb_p):.5f}   ce {roc_auc_score(val.label, val.ce_logit):.5f}")
top = val.sort_values("ce_logit", ascending=False).groupby("rid").head(1)
topx = val.sort_values("xgb_p", ascending=False).groupby("rid").head(1)
has = val.groupby("rid").label.max()
m = has[has == 1].index
print(f"[{TAG}] top-1 correct on records with a match:  ce {top.set_index('rid').loc[m].label.mean():.4f}   "
      f"xgb {topx.set_index('rid').loc[m].label.mean():.4f}")